In [4]:
import numpy as np
import pandas as pd
from md_Helpers import open_run

# ============================================================
# Frames to compare
# Frame IDs are GSD indices: 0 = initial, -1 = final
# ============================================================

RUN_ID_A = "20260903202814"
FRAME_ID_A = -1

RUN_ID_B = "20260903203452"
FRAME_ID_B = -1

# Numerical tolerance
ATOL = 1e-7
RTOL = 0.0


# ============================================================
# Load frames
# ============================================================

frame_a = open_run(RUN_ID_A).load_frame(FRAME_ID_A)
frame_b = open_run(RUN_ID_B).load_frame(FRAME_ID_B)


# ============================================================
# Comparison helpers
# ============================================================

def compare_numeric(name, value_a, value_b):
    value_a = np.asarray(value_a)
    value_b = np.asarray(value_b)

    same_shape = value_a.shape == value_b.shape

    if same_shape:
        exact = np.array_equal(value_a, value_b, equal_nan=True)
        within_tolerance = np.allclose(
            value_a,
            value_b,
            atol=ATOL,
            rtol=RTOL,
            equal_nan=True,
        )

        difference = np.abs(value_a - value_b)
        finite_difference = difference[np.isfinite(difference)]

        max_difference = (
            float(np.max(finite_difference))
            if finite_difference.size
            else 0.0
        )
    else:
        exact = False
        within_tolerance = False
        max_difference = np.nan

    return {
        "Field": name,
        "Shape_A": str(value_a.shape),
        "Shape_B": str(value_b.shape),
        "Same_Shape": same_shape,
        "Exactly_Equal": exact,
        "Within_Tolerance": within_tolerance,
        "Max_Abs_Difference": max_difference,
    }


def compare_text(name, value_a, value_b):
    exact = list(value_a) == list(value_b)

    return {
        "Field": name,
        "Shape_A": str(len(value_a)),
        "Shape_B": str(len(value_b)),
        "Same_Shape": len(value_a) == len(value_b),
        "Exactly_Equal": exact,
        "Within_Tolerance": exact,
        "Max_Abs_Difference": np.nan,
    }


# ============================================================
# Compare state information
# ============================================================

results = [
    compare_numeric(
        "N_Particles",
        [frame_a.particles.N],
        [frame_b.particles.N],
    ),
    compare_numeric(
        "Box",
        frame_a.configuration.box,
        frame_b.configuration.box,
    ),
    compare_numeric(
        "Positions",
        frame_a.particles.position,
        frame_b.particles.position,
    ),
    compare_numeric(
        "Velocities",
        frame_a.particles.velocity,
        frame_b.particles.velocity,
    ),
    compare_text(
        "Particle_Types",
        frame_a.particles.types,
        frame_b.particles.types,
    ),
    compare_numeric(
        "Type_IDs",
        frame_a.particles.typeid,
        frame_b.particles.typeid,
    ),
    compare_numeric(
        "Periodic_Images",
        frame_a.particles.image,
        frame_b.particles.image,
    ),
    compare_numeric(
        "Masses",
        frame_a.particles.mass,
        frame_b.particles.mass,
    ),
    compare_numeric(
        "Orientations",
        frame_a.particles.orientation,
        frame_b.particles.orientation,
    ),
]

comparison = pd.DataFrame(results)

same_physical_state = bool(comparison["Within_Tolerance"].all())
exactly_identical = bool(comparison["Exactly_Equal"].all())

print(f"A: Run {RUN_ID_A}, frame {FRAME_ID_A}, step {frame_a.configuration.step}")
print(f"B: Run {RUN_ID_B}, frame {FRAME_ID_B}, step {frame_b.configuration.step}")
print()
print("Exactly identical:", exactly_identical)
print(f"Same within tolerance ({ATOL:g}):", same_physical_state)

display(comparison)

A: Run 20260903202814, frame -1, step 200000
B: Run 20260903203452, frame -1, step 200000

Exactly identical: False
Same within tolerance (1e-07): False


,Field,Shape_A,Shape_B,Same_Shape,Exactly_Equal,Within_Tolerance,Max_Abs_Difference
0,N_Particles,"(1,)","(1,)",True,True,True,0.000000
1,Box,"(6,)","(6,)",True,True,True,0.000000
2,Positions,"(108000, 3)","(108000, 3)",True,False,False,53.579964
3,Velocities,"(108000, 3)","(108000, 3)",True,False,False,6.469612
4,Particle_Types,1,1,True,True,True,NaN
5,Type_IDs,"(108000,)","(108000,)",True,True,True,0.000000
6,Periodic_Images,"(108000, 3)","(108000, 3)",True,False,False,2.000000
7,Masses,"(108000,)","(108000,)",True,True,True,0.000000
8,Orientations,"(108000, 4)","(108000, 4)",True,True,True,0.000000
